# Schema Introspection and Programmatic Schema Building in LinkML

**ICBO 2025 Tutorial**

This notebook demonstrates:
- Schema introspection with SchemaView
- Practical example: Mapping Biolink predicates to RO terms
- Programmatic schema creation with SchemaBuilder
- Example: Building a microbiome research schema from multiple sources

## Setup and Installation

First, install required packages:

In [1]:
# Uncomment and run if packages not installed
# !pip install linkml linkml-runtime biolink-model pandas

In [2]:
# Import required libraries
from importlib.resources import files
from linkml_runtime.utils.schemaview import SchemaView
from linkml.utils.schema_builder import SchemaBuilder
from linkml_runtime.dumpers import yaml_dumper
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

---
# Part 1: Schema Introspection with SchemaView

SchemaView is LinkML's primary tool for dynamically introspecting and manipulating schemas.

## Loading Schemas

SchemaView can load schemas from:
- Installed Python packages
- URLs
- Local files

In [3]:
# Load Biolink Model from installed package (recommended)
biolink_yaml = files('biolink_model.schema') / 'biolink_model.yaml'
sv = SchemaView(str(biolink_yaml))

print(f"Loaded schema: {sv.schema.name}")
print(f"Version: {sv.schema.version}")
print(f"Description: {sv.schema.description}")

Loaded schema: Biolink-Model
Version: 4.2.5
Description: Entity and association taxonomy and datamodel for life-sciences data


## Basic Schema Exploration

In [4]:
# Get all elements
all_classes = sv.all_classes()
all_slots = sv.all_slots()
all_enums = sv.all_enums()

print(f"Total classes: {len(all_classes)}")
print(f"Total slots: {len(all_slots)}")
print(f"Total enums: {len(all_enums)}")

# Show some examples
print(f"\nExample classes: {list(all_classes)[:5]}")
print(f"Example slots: {list(all_slots)[:5]}")

Total classes: 308
Total slots: 474
Total enums: 22

Example classes: ['mapping collection', 'predicate mapping', 'ontology class', 'annotation', 'quantity value']
Example slots: ['has attribute', 'has attribute type', 'has qualitative value', 'has quantitative value', 'has numeric value']


In [5]:
# Get specific elements
gene_class = sv.get_class("gene")
print(f"Gene class: {gene_class.name}")
print(f"Description: {gene_class.description}")
print(f"Is a: {gene_class.is_a}")

related_to_slot = sv.get_slot("related to")
print(f"\nSlot: {related_to_slot.name}")
print(f"Description: {related_to_slot.description}")

Gene class: gene
Description: A region (or regions) that includes all of the sequence elements necessary to encode a functional transcript. A gene locus may include regulatory regions, transcribed regions and/or other functional sequence regions.
Is a: biological entity

Slot: related to
Description: A relationship that is asserted between two named things


## Navigating Hierarchies

SchemaView provides powerful methods for navigating class and slot hierarchies.

In [6]:
# Class hierarchy navigation
print("gene hierarchy:")
print(f"  Parents: {sv.class_parents('gene')}")
print(f"  Children: {sv.class_children('gene')[:5]}")  # First 5
print(f"  Ancestors: {sv.class_ancestors('gene')}")
print(f"  Descendants count: {len(sv.class_descendants('gene'))}")

gene hierarchy:
  Parents: ['gene or gene product', 'genomic entity', 'chemical entity or gene or gene product', 'physical essence', 'ontology class', 'biological entity']
  Children: []
  Ancestors: ['gene', 'gene or gene product', 'genomic entity', 'chemical entity or gene or gene product', 'physical essence', 'ontology class', 'biological entity', 'thing with taxon', 'named thing', 'entity', 'physical essence or occurrent', 'macromolecular machine mixin']
  Descendants count: 1


In [7]:
# Find root and leaf classes
roots = sv.class_roots()
leaves = sv.class_leaves()

print(f"Root classes: {roots}")
print(f"\nNumber of leaf classes: {len(leaves)}")
print(f"Example leaf classes: {leaves[:10]}")

Root classes: ['mapping collection', 'predicate mapping', 'ontology class', 'annotation', 'relationship quantifier', 'chemical or drug or treatment', 'entity', 'physical essence or occurrent', 'subject of investigation', 'thing with taxon', 'genomic entity', 'epigenomic entity', 'chemical entity or gene or gene product', 'chemical entity or protein or polypeptide', 'macromolecular machine mixin', 'gene grouping mixin', 'pathological entity mixin', 'outcome', 'gene expression mixin', 'cell line to entity association mixin', 'chemical entity to entity association mixin', 'case to entity association mixin', 'material sample to entity association mixin', 'disease to entity association mixin', 'entity to exposure event association mixin', 'entity to outcome association mixin', 'frequency qualifier mixin', 'disease or phenotypic feature to entity association mixin', 'entity to disease or phenotypic feature association mixin', 'genotype to entity association mixin', 'gene to entity associatio

## Example: Extracting Biolink to RO Mappings

Let's extract SKOS-style relationships between Biolink predicates and RO terms.

In [8]:
# Get all descendants of "related to" slot
related_to_descendants = sv.slot_descendants("related to")

print(f"Found {len(related_to_descendants)} predicates descended from 'related to'")
print(f"\nExamples: {list(related_to_descendants)[:10]}")

Found 243 predicates descended from 'related to'

Examples: ['related to', 'related to at concept level', 'related to at instance level', 'disease has location', 'location of disease', 'composed primarily of', 'primarily composed of', 'associated with', 'opposite of', 'affects likelihood of']


In [9]:
def get_mappings(sv, slot):
    """Get all mappings for a single slot"""
    mapping_types = {
        'exact_mappings': 'skos:exactMatch',
        'broad_mappings': 'skos:broadMatch',
        'narrow_mappings': 'skos:narrowMatch',
        'related_mappings': 'skos:relatedMatch'
    }
    
    results = []
    for mapping_type, skos_pred in mapping_types.items():
        mappings = getattr(slot, mapping_type, [])
        for term in mappings:
            results.append({
                'biolink_predicate': sv.get_uri(slot, expand=False),
                'mapping_type': skos_pred,
                'mapped_term': term
            })
    
    return results

# Example: single slot
slot = sv.get_slot("has phenotype")
mappings = get_mappings(sv, slot)
print(f"Mappings for 'has phenotype':")
for m in mappings:
    print(f"  {m}")

Mappings for 'has phenotype':
  {'biolink_predicate': 'biolink:has_phenotype', 'mapping_type': 'skos:exactMatch', 'mapped_term': 'RO:0002200'}
  {'biolink_predicate': 'biolink:has_phenotype', 'mapping_type': 'skos:broadMatch', 'mapped_term': 'NCIT:R115'}
  {'biolink_predicate': 'biolink:has_phenotype', 'mapping_type': 'skos:broadMatch', 'mapped_term': 'NCIT:R108'}
  {'biolink_predicate': 'biolink:has_phenotype', 'mapping_type': 'skos:narrowMatch', 'mapped_term': 'NCIT:R89'}
  {'biolink_predicate': 'biolink:has_phenotype', 'mapping_type': 'skos:narrowMatch', 'mapped_term': 'DOID-PROPERTY:has_symptom'}
  {'biolink_predicate': 'biolink:has_phenotype', 'mapping_type': 'skos:narrowMatch', 'mapped_term': 'RO:0004022'}
  {'biolink_predicate': 'biolink:has_phenotype', 'mapping_type': 'skos:narrowMatch', 'mapped_term': 'RO:0004029'}


In [10]:
# Apply to all descendants of "related to"
all_results = []

for slot_id in sv.slot_descendants("related to", reflexive=True):
    slot = sv.get_slot(slot_id)
    all_results.extend(get_mappings(sv, slot))

# Convert to DataFrame and filter for RO terms
df = pd.DataFrame(all_results)
df = df[df['mapped_term'].str.startswith('RO:')]
df = df.rename(columns={'mapped_term': 'ro_term'})

# Shuffle to show variety of predicates
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# View results
print(f"Found {len(df)} RO mappings\n")
print(df.head(10).to_string(index=False))

Found 200 RO mappings

            biolink_predicate     mapping_type       ro_term
biolink:temporally_related_to skos:narrowMatch    RO:0002092
           biolink:related_to skos:narrowMatch    RO:0002179
           biolink:related_to skos:narrowMatch    RO:0002373
       biolink:orthologous_to  skos:exactMatch RO:HOM0000017
            biolink:has_input skos:narrowMatch    RO:0002590
             biolink:precedes skos:narrowMatch    RO:0002412
            biolink:occurs_in skos:narrowMatch    RO:0002231
            biolink:caused_by skos:narrowMatch    RO:0009501
               biolink:causes skos:narrowMatch    RO:0002256
      biolink:associated_with skos:narrowMatch    RO:0004029


In [11]:
# Group by mapping type
mapping_summary = df.groupby('mapping_type').size().sort_values(ascending=False)

print("SKOS mapping types distribution:\n")
max_count = mapping_summary.max()
for mapping_type, count in mapping_summary.items():
    bar = '█' * int(50 * count / max_count)
    print(f"{mapping_type:20s} {bar} {count}")

print("\nInterpretation (SKOS semantics):")
print("  - skos:exactMatch: Direct semantic equivalence")
print("  - skos:broadMatch: RO term is more general")
print("  - skos:narrowMatch: RO term is more specific")
print("  - skos:relatedMatch: Related but not hierarchical")

SKOS mapping types distribution:

skos:narrowMatch     ██████████████████████████████████████████████████ 134
skos:exactMatch      █████████████████████ 57
skos:broadMatch      ███ 9

Interpretation (SKOS semantics):
  - skos:exactMatch: Direct semantic equivalence
  - skos:broadMatch: RO term is more general
  - skos:narrowMatch: RO term is more specific
  - skos:relatedMatch: Related but not hierarchical


## Advanced SchemaView Features

In [12]:
# Get all slots applicable to a class
gene_slots = sv.class_slots("gene")
print(f"Slots for gene class: {len(gene_slots)}")
print(f"Examples: {gene_slots[:10]}")

Slots for gene class: 16
Examples: ['symbol', 'xref', 'has biological sequence', 'id', 'in taxon', 'in taxon label', 'provided by', 'full name', 'synonym', 'iri']


In [13]:
# Get URIs with prefix expansion
gene_uri = sv.get_uri("gene", expand=True)
print(f"Gene URI: {gene_uri}")

related_to_uri = sv.get_uri("related to", expand=True)
print(f"'related to' URI: {related_to_uri}")

ChemBank namespace is already mapped to http://identifiers.org/chembank/ - Overriding with mapping to http://chembank.broadinstitute.org/chemistry/viewMolecule.htm?cbid=


CLINVAR namespace is already mapped to http://identifiers.org/clinvar/ - Overriding with mapping to http://identifiers.org/clinvar


ComplexPortal namespace is already mapped to http://identifiers.org/complexportal/ - Overriding with mapping to https://www.ebi.ac.uk/complexportal/complex/


CTD.CHEMICAL namespace is already mapped to http://identifiers.org/ctd.chemical/ - Overriding with mapping to http://ctdbase.org/detail.go?type=chem&acc=


CTD.DISEASE namespace is already mapped to http://identifiers.org/ctd.disease/ - Overriding with mapping to http://ctdbase.org/detail.go?type=disease&db=MESH&acc=


...

WBbt namespace is already mapped to http://purl.obolibrary.org/obo/WBBT_ - Overriding with mapping to http://purl.obolibrary.org/obo/WBbt_


WIKIDATA namespace is already mapped to http://identifiers.org/wikidata/ - Overriding with mapping to https://www.wikidata.org/entity/


Xenbase namespace is already mapped to http://identifiers.org/xenbase/ - Overriding with mapping to http://www.xenbase.org/gene/showgene.do?method=display&geneId=


PMC namespace is already mapped to http://identifiers.org/pmc/ - Overriding with mapping to http://europepmc.org/articles/PMC


Gene URI: https://w3id.org/biolink/vocab/Gene
'related to' URI: https://w3id.org/biolink/vocab/related_to


In [14]:
# Find inverse predicates
has_part_inverse = sv.inverse("has part")
print(f"Inverse of 'has part': {has_part_inverse}")

part_of_inverse = sv.inverse("part of")
print(f"Inverse of 'part of': {part_of_inverse}")

Inverse of 'has part': part of
Inverse of 'part of': has part


In [15]:
# Induced slots - slots can be refined in subclasses via slot_usage
# Example: "subject" slot gets narrowed in gene to gene association

# The base slot has a broad range
base_subject = sv.get_slot("subject")
print(f"Base 'subject' slot:")
print(f"  Range: {base_subject.range}")
print(f"  Description: {base_subject.description[:80]}...")

# But in "gene to gene association", it's refined via slot_usage
induced = sv.induced_slot("subject", "gene to gene association")
print(f"\nInduced 'subject' slot in 'gene to gene association':")
print(f"  Range: {induced.range}")
print(f"  Description: {induced.description[:80]}...")

Base 'subject' slot:
  Range: named thing
  Description: connects an association to the subject of the association. For example, in a gen...

Induced 'subject' slot in 'gene to gene association':
  Range: gene or gene product
  Description: the subject gene in the association. If the relation is symmetric, subject vs ob...


---
# Part 2: Programmatic Schema Building with SchemaBuilder

SchemaBuilder enables programmatic schema creation and composition from multiple sources.

## Example: Building a Microbiome Research Schema

We'll create a focused schema by combining:
- **Biolink** entity classes
- **RO terms** for relationships
- **NMDC schema** components
- **Custom** domain-specific additions

### Step 1: Load Source Schemas

In [16]:
# Load Biolink Model (already loaded as 'sv' above)
biolink_sv = sv

# Load NMDC schema from GitHub
print("Loading NMDC schema...")
nmdc_sv = SchemaView("https://raw.githubusercontent.com/microbiomedata/nmdc-schema/main/src/schema/nmdc.yaml")
print(f"Loaded NMDC schema: {nmdc_sv.schema.name}")
print(f"NMDC classes: {len(nmdc_sv.all_classes())}")
print(f"NMDC slots: {len(nmdc_sv.all_slots())}")

Loading NMDC schema...


Loaded NMDC schema: NMDC


NMDC classes: 76
NMDC slots: 851


In [17]:
# Create new schema with SchemaBuilder
sb = SchemaBuilder("microbiome-research-schema")
sb.add_defaults()

# Add prefixes
sb.add_prefix("biolink", "https://w3id.org/biolink/vocab/")
sb.add_prefix("RO", "http://purl.obolibrary.org/obo/RO_")
sb.add_prefix("nmdc", "https://w3id.org/nmdc/")
sb.add_prefix("ex", "https://example.org/microbiome/")

print("Created new schema: microbiome-research-schema")

Created new schema: microbiome-research-schema


### Step 2: Extract Classes from Biolink

In [18]:
# Get relevant Biolink entity classes
biolink_classes = ["organism taxon", "gene", "protein", 
                   "chemical entity", "biological entity"]

print("Extracting Biolink classes:")
for class_name in biolink_classes:
    cls = biolink_sv.get_class(class_name)
    if cls:
        # Use title case for our new schema
        new_class_name = class_name.title().replace(" ", "")
        sb.add_class(
            new_class_name,
            description=cls.description,
            class_uri=f"biolink:{class_name.replace(' ', '')}",
            slots=["id", "name", "category"]
        )
        print(f"  ✓ {class_name} → {new_class_name}")
    else:
        print(f"  ✗ {class_name} not found")

Extracting Biolink classes:
  ✓ organism taxon → OrganismTaxon
  ✓ gene → Gene
  ✓ protein → Protein
  ✓ chemical entity → ChemicalEntity
  ✓ biological entity → BiologicalEntity


### Step 3: Extract Classes from NMDC

In [19]:
# Get NMDC-specific classes for environmental samples
nmdc_classes = ["Biosample", "Study", "ProcessedSample"]

print("Extracting NMDC classes:")
for class_name in nmdc_classes:
    cls = nmdc_sv.get_class(class_name)
    if cls:
        # Get some key slots from the NMDC class
        key_slots = ["id", "name", "description", "type"]
        
        sb.add_class(
            class_name,
            description=cls.description or f"NMDC {class_name}",
            class_uri=f"nmdc:{class_name}",
            slots=key_slots
        )
        print(f"  ✓ {class_name}")
    else:
        print(f"  ✗ {class_name} not found")

Extracting NMDC classes:
  ✓ Biosample
  ✓ Study
  ✓ ProcessedSample


### Step 4: Add RO-based Relationships

In [20]:
# Define microbiome-relevant relationships from RO
ro_relationships = {
    "inhabits": {
        "description": "A relationship between an organism and its habitat",
        "uri": "RO:0002314",  # ecologically related to
        "domain": "OrganismTaxon",
        "range": "Biosample"
    },
    "derived_from": {
        "description": "Sample is derived from another sample",
        "uri": "RO:0001000",
        "domain": "ProcessedSample",
        "range": "Biosample"
    },
    "has_part": {
        "description": "Entity has another entity as a part",
        "uri": "RO:0001019",
        "domain": "Biosample",
        "range": "OrganismTaxon"
    },
    "produces": {
        "description": "Organism produces a chemical entity",
        "uri": "RO:0003000",
        "domain": "OrganismTaxon",
        "range": "ChemicalEntity"
    }
}

print("Adding RO-based relationship slots:")
for slot_name, props in ro_relationships.items():
    sb.add_slot(slot_name,
                description=props["description"],
                domain=props["domain"],
                range=props["range"],
                slot_uri=props["uri"])
    print(f"  ✓ {slot_name} ({props['uri']})")

Adding RO-based relationship slots:
  ✓ inhabits (RO:0002314)
  ✓ derived_from (RO:0001000)
  ✓ has_part (RO:0001019)
  ✓ produces (RO:0003000)


### Step 5: Add Custom Domain Components

In [21]:
# Add domain-specific enumerations
sb.add_enum("SampleType",
            permissible_values={
                "soil": {"description": "Soil sample"},
                "water": {"description": "Water sample"},
                "sediment": {"description": "Sediment sample"},
                "host-associated": {"description": "Sample from host organism"}
            })

print("Added SampleType enum")

# Add custom type
sb.add_type("ISODateTime",
            typeof="string",
            description="ISO 8601 formatted datetime",
            pattern="^\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}:\\d{2}")

print("Added ISODateTime type")

Added SampleType enum
Added ISODateTime type


In [22]:
# Add custom slots FIRST (before the class that uses them)
sb.add_slot("sequencing_platform",
            description="Platform used for sequencing",
            range="string")

sb.add_slot("total_reads",
            description="Total number of sequencing reads",
            range="integer")

sb.add_slot("analysis_date",
            description="Date when analysis was performed",
            range="ISODateTime")

sb.add_slot("sample_id",
            description="Reference to the biosample",
            range="Biosample")

print("Added custom slots")

# NOW add the class that uses these slots
sb.add_class("MetagenomicAnalysis",
             description="A metagenomic sequencing analysis",
             slots=["id", "sample_id", "sequencing_platform", 
                    "total_reads", "analysis_date"],
             class_uri="ex:MetagenomicAnalysis")

print("Added MetagenomicAnalysis class")

Added custom slots
Added MetagenomicAnalysis class


### Step 6: Export and Introspect the Composed Schema

In [23]:
# Get the completed schema
schema = sb.schema

print(f"Schema name: {schema.name}")
print(f"Schema ID: {schema.id}")

Schema name: microbiome-research-schema
Schema ID: http://example.org/microbiome-research-schema


In [24]:
# Export to YAML file
output_file = "microbiome-research-schema.yaml"
yaml_dumper.dump(schema, output_file)
print(f"Exported schema to: {output_file}")

Exported schema to: microbiome-research-schema.yaml


In [25]:
# Now use SchemaView to introspect our new composed schema!
composed_sv = SchemaView(schema)

print("\n=== Composed Schema Summary ===")
print(f"Classes: {len(composed_sv.all_classes())}")
print(f"Slots: {len(composed_sv.all_slots())}")
print(f"Enums: {len(composed_sv.all_enums())}")
print(f"Types: {len(composed_sv.all_types())}")

print(f"\nAll classes: {list(composed_sv.all_classes())}")


=== Composed Schema Summary ===
Classes: 9
Slots: 13
Enums: 1
Types: 20

All classes: ['OrganismTaxon', 'Gene', 'Protein', 'ChemicalEntity', 'BiologicalEntity', 'Biosample', 'Study', 'ProcessedSample', 'MetagenomicAnalysis']


In [26]:
# Find all RO-mapped relationships
ro_slots = [s for s in composed_sv.all_slots()
            if composed_sv.get_slot(s).slot_uri 
            and "RO:" in composed_sv.get_slot(s).slot_uri]

print("RO-mapped relationships in our schema:")
for slot_name in ro_slots:
    slot = composed_sv.get_slot(slot_name)
    print(f"  {slot_name}: {slot.slot_uri}")
    print(f"    {slot.description}")

RO-mapped relationships in our schema:
  inhabits: RO:0002314
    A relationship between an organism and its habitat
  derived_from: RO:0001000
    Sample is derived from another sample
  has_part: RO:0001019
    Entity has another entity as a part
  produces: RO:0003000
    Organism produces a chemical entity


In [27]:
# Examine the MetagenomicAnalysis class we created
mga_class = composed_sv.get_class("MetagenomicAnalysis")
print(f"MetagenomicAnalysis class:")
print(f"  Description: {mga_class.description}")
print(f"  URI: {mga_class.class_uri}")
print(f"  Slots: {mga_class.slots}")

MetagenomicAnalysis class:
  Description: A metagenomic sequencing analysis
  URI: ex:MetagenomicAnalysis
  Slots: ['id', 'sample_id', 'sequencing_platform', 'total_reads', 'analysis_date']


In [28]:
# Show the SampleType enum
sample_type_enum = composed_sv.get_enum("SampleType")
print("SampleType enum values:")
for pv_name, pv_obj in sample_type_enum.permissible_values.items():
    print(f"  {pv_name}: {pv_obj.description}")

SampleType enum values:
  soil: None
  water: None
  sediment: None
  host-associated: None


## Key Takeaway: SchemaBuilder → SchemaView Round-trip

Notice how we:
1. Used **SchemaView** to introspect existing schemas (Biolink, NMDC)
2. Used **SchemaBuilder** to compose a new schema from multiple sources
3. Used **SchemaView** again to introspect our newly created schema

This round-trip pattern enables **iterative schema development** and validation!

---
## Summary

**SchemaView:**
- Essential for schema introspection and analysis
- Navigate complex hierarchies with ease
- Extract mappings and semantic relationships
- Foundation for schema-driven applications

**SchemaBuilder:**
- Programmatically create and modify schemas
- Combine components from multiple sources
- Implement schema transformations
- Generate schemas from external data

**Together:** Powerful tools for working with semantic data models in the life sciences!